# Dysarthria / ALS Speech Detection — HuBERT + LoRA (Anti-Overfitting Edition)
## TORGO Dataset · Colab T4 Safe · Comprehensive Metrics Output

### Anti-Overfitting Fixes Applied
| Problem | Fix |
|---|---|
| Weak regularisation in head | Dropout raised to 0.6, L2 weight_decay = 5e-2 |
| No audio augmentation | Time-stretch, pitch-shift, additive noise, time-masking |
| LoRA rank too low (r=4) | r=8, lora_alpha=16 for better capacity |
| LoRA dropout too low | lora_dropout raised to 0.2 |
| BCE loss no smoothing | Label smoothing ε=0.1 via custom LabelSmoothBCE |
| Patience=2 too aggressive | PATIENCE=5, monitor val AUC-ROC not F1 |
| Missing metrics | Full metrics export: AUC-ROC, PR-AUC, MCC, per-speaker, CSV/JSON |
| No LR warmup tuning | Warmup = 15% of total steps |


In [1]:
# Do NOT reinstall torch — Colab's pre-installed version is correct
!pip install -q transformers==4.40.2 peft==0.10.0 accelerate librosa soundfile scikit-learn matplotlib seaborn
print("Install complete. Now do Runtime → Restart session, then run from Cell 2.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 55.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.2 which is incompatible.
Install complete. Now do Runtime → Restart session, then run from Cell 2.


In [1]:
import os, re, gc, warnings, json, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import librosa
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
warnings.filterwarnings("ignore")

from transformers import HubertModel, Wav2Vec2FeatureExtractor
from peft import LoraConfig, inject_adapter_in_model
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_recall_curve,
    roc_auc_score, roc_curve, average_precision_score,
    matthews_corrcoef
)
from torch.cuda.amp import autocast, GradScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {device}")
if device.type == "cuda":
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.12/dist-package

PyTorch : 2.2.2+cu121
Device  : cpu


In [2]:
!kaggle datasets download -d pranaykoppula/torgo-audio -q
!unzip -o torgo-audio.zip -d /content/torgo > /dev/null
print("Dataset ready.")


Dataset URL: https://www.kaggle.com/datasets/pranaykoppula/torgo-audio
License(s): other
Dataset ready.


In [3]:
def load_data(dataset_path="/content/torgo"):
    file_paths, labels = [], []
    for root, dirs, files in os.walk(dataset_path):
        for file in files:
            if not file.endswith(".wav"):
                continue
            path = os.path.join(root, file)
            rl = root.lower()
            if "dys" in rl:
                labels.append(1)
            elif "con" in rl:
                labels.append(0)
            else:
                continue
            file_paths.append(path)
    return file_paths, np.array(labels)

file_paths, labels = load_data("/content/torgo")
if len(file_paths) == 0:
    file_paths, labels = load_data("/content")

print(f"Total      : {len(file_paths)}")
print(f"Control (0): {np.sum(labels==0)}")
print(f"Dysarth (1): {np.sum(labels==1)}")


Total      : 17635
Control (0): 11456
Dysarth (1): 6179


In [4]:
# Test  : M01 (severe), M05 (mild), FC01, MC04
# Val   : F01 (severe), FC02   ← no speaker leakage into train
# Train : everyone else

TEST_SPEAKERS = {"M01", "M05", "FC01", "MC04"}
VAL_SPEAKERS  = {"F01", "FC02"}

def get_speaker(path):
    folder = os.path.basename(os.path.dirname(path))
    m = re.search(r"(MC\d+|FC\d+|M\d+|F\d+)", folder)
    return m.group(1) if m else "UNK"

X_train, y_train = [], []
X_val,   y_val   = [], []
X_test,  y_test  = [], []

for p, l in zip(file_paths, labels):
    s = get_speaker(p)
    if   s in TEST_SPEAKERS: X_test.append(p);  y_test.append(l)
    elif s in VAL_SPEAKERS:  X_val.append(p);   y_val.append(l)
    else:                    X_train.append(p); y_train.append(l)

y_train, y_val, y_test = map(np.array, [y_train, y_val, y_test])

for split, y in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    print(f"{split:5s}: {len(y):5d}  ctrl={np.sum(y==0)}  dys={np.sum(y==1)}")


Train: 11588  ctrl=7100  dys=4488
Val  :  2529  ctrl=2261  dys=268
Test :  3518  ctrl=2095  dys=1423


In [5]:
SR          = 16000
MAX_SAMPLES = SR * 5

def preprocess_audio(path, augment=False):
    """Load and optionally augment audio.

    Augmentations applied only during training (augment=True):
    - Random time stretch  (rate in [0.9, 1.1])
    - Random pitch shift   (semitones in [-2, 2])
    - Additive Gaussian noise (SNR ~20 dB)
    - Random time masking  (mask up to 10% of frames)
    """
    try:
        audio, _ = librosa.load(path, sr=SR, mono=True)
        if len(audio) < 400:
            return None
        # Normalise before augmentation so noise level is consistent
        audio = audio / (np.max(np.abs(audio)) + 1e-8)

        if augment:
            # 1. Time stretch
            if random.random() < 0.4:
                rate = random.uniform(0.9, 1.1)
                audio = librosa.effects.time_stretch(audio, rate=rate)

            # 2. Pitch shift
            if random.random() < 0.4:
                n_steps = random.uniform(-2.0, 2.0)
                audio = librosa.effects.pitch_shift(audio, sr=SR, n_steps=n_steps)

            # 3. Additive Gaussian noise (~20 dB SNR)
            if random.random() < 0.5:
                noise_amp = 0.005 * np.random.randn(len(audio))
                audio = audio + noise_amp

            # 4. Time masking — zero out a random contiguous segment
            if random.random() < 0.3:
                mask_len   = int(random.uniform(0.0, 0.10) * len(audio))
                mask_start = random.randint(0, max(0, len(audio) - mask_len - 1))
                audio[mask_start : mask_start + mask_len] = 0.0

        # Truncate / pad & final normalise
        audio = audio[:MAX_SAMPLES]
        peak  = np.max(np.abs(audio))
        audio = audio / (peak + 1e-8)

        if np.isnan(audio).any():
            return None
        return audio.astype(np.float32)
    except Exception:
        return None

print(f"Audio utils ready  (SR={SR}, max={MAX_SAMPLES//SR}s, augmentation enabled)")


Audio utils ready  (SR=16000, max=5s, augmentation enabled)


In [6]:
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/hubert-base-ls960")

hubert = HubertModel.from_pretrained("facebook/hubert-base-ls960")
hubert.gradient_checkpointing_enable()

# ── LoRA config: r=8 for more capacity, higher dropout to regularise ──────────
lora_config = LoraConfig(
    r=8,                          # was 4 → doubled rank
    lora_alpha=16,                # was 8  → scaled proportionally
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.2,             # was 0.1 → stronger regularisation
    bias="none",
    task_type=None,
)

hubert_lora = inject_adapter_in_model(lora_config, hubert)

# Freeze backbone; only LoRA params trainable
for name, param in hubert_lora.named_parameters():
    param.requires_grad = "lora_" in name

total     = sum(p.numel() for p in hubert_lora.parameters())
trainable = sum(p.numel() for p in hubert_lora.parameters() if p.requires_grad)
print(f"Total params     : {total:,}")
print(f"Trainable params : {trainable:,}  ({100*trainable/total:.2f}%)")

hubert_lora = hubert_lora.to(device)


# ── Label-smoothed BCE ────────────────────────────────────────────────────────
class LabelSmoothBCELoss(nn.Module):
    """BCEWithLogitsLoss with label smoothing (ε=0.1).

    Smoothed targets: y_smooth = y*(1-ε) + 0.5*ε
    This prevents the model from being over-confident on noisy medical labels.
    """
    def __init__(self, eps=0.1, pos_weight=None):
        super().__init__()
        self.eps = eps
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def forward(self, logits, targets):
        smooth = targets * (1 - self.eps) + 0.5 * self.eps
        return self.bce(logits, smooth)


# ── Classifier head ────────────────────────────────────────────────────────────
class MeanStdPooling(nn.Module):
    def forward(self, x):           # x: (B, T, D)
        mean = x.mean(dim=1)
        std  = x.std(dim=1).clamp(min=1e-6)
        return torch.cat([mean, std], dim=1)   # (B, 2D)


class DysarthriaClassifier(nn.Module):
    """HuBERT + LoRA backbone with regularised classification head.

    Changes vs original:
    - Dropout raised to 0.6 (was 0.5) throughout
    - Added BatchNorm after first linear for stable training
    - Smaller bottleneck (128 instead of 256) to reduce capacity and overfit risk
    """
    def __init__(self, backbone, hidden=768, dropout=0.6):
        super().__init__()
        self.backbone = backbone
        self.pool     = MeanStdPooling()
        self.head = nn.Sequential(
            nn.LayerNorm(hidden * 2),
            nn.Linear(hidden * 2, 128),     # was 256 → smaller bottle-neck
            nn.GELU(),
            nn.Dropout(dropout),            # 0.6
            nn.Linear(128, 32),             # was 64 → further compression
            nn.GELU(),
            nn.Dropout(dropout),            # 0.6
            nn.Linear(32, 1),
        )

    def forward(self, input_values, attention_mask=None):
        out    = self.backbone(input_values=input_values,
                               attention_mask=attention_mask)
        hidden = out.last_hidden_state       # (B, T, 768)
        pooled = self.pool(hidden)           # (B, 1536)
        return self.head(pooled)             # (B, 1)


model = DysarthriaClassifier(hubert_lora).to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal model params     : {total:,}")
print(f"Trainable model params : {trainable:,}  ({100*trainable/total:.2f}%)")


Some weights of the model checkpoint at facebook/hubert-base-ls960 were not used when initializing HubertModel: ['encoder.pos_conv_embed.conv.weight_g', 'encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing HubertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing HubertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of HubertModel were not initialized from the model checkpoint at facebook/hubert-base-ls960 and are newly initialized: ['encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a down-stream task to be able to use it for pre

Total params     : 94,666,624
Trainable params : 294,912  (0.31%)

Total model params     : 94,870,593
Trainable model params : 498,881  (0.53%)


In [7]:
class TorgoDataset(Dataset):
    def __init__(self, paths, labels, augment=False):
        self.paths   = paths
        self.labels  = labels
        self.augment = augment        # ← NEW: passed to preprocess_audio

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        audio = preprocess_audio(self.paths[idx], augment=self.augment)
        return audio, float(self.labels[idx])


def collate_fn(batch):
    batch = [(a, l) for a, l in batch if a is not None]
    if not batch:
        return None, None, None
    audios, labels = zip(*batch)
    audios = [np.array(a, dtype=np.float32) for a in audios]
    enc = feature_extractor(
        audios,
        sampling_rate=SR,
        return_tensors="pt",
        padding=True,
        return_attention_mask=True,
        max_length=MAX_SAMPLES,
        truncation=True,
    )
    return (enc["input_values"],
            enc["attention_mask"],
            torch.tensor(labels, dtype=torch.float32))


BATCH_SIZE = 4

# ── Build datasets (train WITH augmentation) ───────────────────────────────────
dataset_train = TorgoDataset(X_train, y_train, augment=True)   # ← augmented
dataset_val   = TorgoDataset(X_val,   y_val,   augment=False)  # ← clean

# Balance validation set
random.seed(42)
ctrl_idx = [i for i, l in enumerate(y_val) if l == 0]
dys_idx  = [i for i, l in enumerate(y_val) if l == 1]
n_min    = min(len(ctrl_idx), len(dys_idx))
balanced_idx = random.sample(ctrl_idx, n_min) + random.sample(dys_idx, n_min)
random.shuffle(balanced_idx)
val_dataset_balanced = Subset(dataset_val, balanced_idx)

print(f"Full val     : ctrl={len(ctrl_idx)}  dys={len(dys_idx)}")
print(f"Balanced val : ctrl={n_min}  dys={n_min}  total={len(val_dataset_balanced)}")

train_loader = DataLoader(dataset_train, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=0, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset_balanced, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0, collate_fn=collate_fn)
test_loader  = DataLoader(TorgoDataset(X_test, y_test, augment=False),
                          batch_size=8, shuffle=False,
                          collate_fn=collate_fn, num_workers=0, pin_memory=False)

print(f"\nTrain batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")


Full val     : ctrl=2261  dys=268
Balanced val : ctrl=268  dys=268  total=536

Train batches : 2897
Val   batches : 134
Test  batches : 440


In [10]:
!pip install torch==2.2.2
!pip install sympy>=1.12
!pip install transformers==4.40.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 88.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.40.2
    Uninstalling transformers-4.40.2:
      Successfully uninstalled transformers-4.40.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.


In [8]:

EPOCHS        = 10            # more epochs; early-stop will trim
PATIENCE      = 5                 # was 2 → give model more time to generalise
patience_counter = 0

# ── Compute pos_weight for class imbalance ────────────────────────────────────
n_ctrl = np.sum(y_train == 0)
n_dys  = np.sum(y_train == 1)
pos_w  = torch.tensor([n_ctrl / (n_dys + 1e-8)], dtype=torch.float32).to(device)
print(f"pos_weight = {pos_w.item():.3f}  (ctrl={n_ctrl}, dys={n_dys})")

criterion = LabelSmoothBCELoss(eps=0.1, pos_weight=pos_w)

# ── Stronger L2 via weight_decay ──────────────────────────────────────────────
optimizer = optim.AdamW(model.parameters(), lr=3e-5,   # was 5e-5 → lower LR
                        weight_decay=5e-2)              # was 1e-2 → 5× stronger

from transformers import get_cosine_schedule_with_warmup
total_steps  = EPOCHS * len(train_loader)
warmup_steps = int(0.15 * total_steps)                 # was 10% → 15%
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
scaler = GradScaler()

train_losses, val_losses = [], []
val_aucs = []
best_auc = 0.0          # monitor AUC-ROC (less threshold-dependent than F1)
best_epoch = 0

history = []            # full per-epoch record for CSV export

for epoch in range(EPOCHS):
    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    t_loss, t_steps = 0.0, 0

    for iv, am, lbl in train_loader:
        if iv is None:
            continue
        iv  = iv.to(device)
        am  = am.to(device)
        lbl = lbl.unsqueeze(1).to(device)

        optimizer.zero_grad(set_to_none=True)
        with autocast():
            logits = model(iv, am)
            loss   = criterion(logits, lbl)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        t_loss  += loss.item()
        t_steps += 1

    avg_train = t_loss / max(t_steps, 1)
    train_losses.append(avg_train)

    # ── Validate ──────────────────────────────────────────────────────────────
    model.eval()
    v_loss, v_steps = 0.0, 0
    all_probs, all_true = [], []

    with torch.no_grad():
        for iv, am, lbl in val_loader:
            if iv is None:
                continue
            iv  = iv.to(device)
            am  = am.to(device)
            lbl = lbl.unsqueeze(1).to(device)
            with autocast():
                logits = model(iv, am)
                v_loss += criterion(logits, lbl).item()
            v_steps += 1
            all_probs.extend(torch.sigmoid(logits).cpu().numpy().flatten())
            all_true.extend(lbl.cpu().numpy().flatten())

    avg_val   = v_loss / max(v_steps, 1)
    val_losses.append(avg_val)

    all_probs_np = np.array(all_probs)
    all_true_np  = np.array(all_true)

    preds   = (all_probs_np > 0.5).astype(int)
    val_acc = accuracy_score(all_true_np, preds)
    val_f1  = f1_score(all_true_np, preds, zero_division=0)
    try:
        val_auc = roc_auc_score(all_true_np, all_probs_np)
    except ValueError:
        val_auc = 0.5
    val_aucs.append(val_auc)

    # overfit gap metric
    gap = avg_train - avg_val   # negative = val loss < train loss (good)

    rec = dict(epoch=epoch+1, train_loss=avg_train, val_loss=avg_val,
               val_acc=val_acc, val_f1=val_f1, val_auc=val_auc, gap=gap)
    history.append(rec)

    print(f"Epoch {epoch+1:2d}/{EPOCHS}  "
          f"train={avg_train:.4f}  val={avg_val:.4f}  "
          f"gap={gap:+.4f}  acc={val_acc:.3f}  f1={val_f1:.3f}  auc={val_auc:.3f}")

    if val_auc > best_auc:
        best_auc = val_auc
        best_epoch = epoch + 1
        patience_counter = 0
        torch.save(hubert_lora.state_dict(), "best_lora_adapter.pt")
        torch.save(model.head.state_dict(),  "best_head.pt")
        print(f"  ✓ Saved best checkpoint  (auc={best_auc:.4f})")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

    torch.cuda.empty_cache()
    gc.collect()

print(f"\nTraining complete.  Best val AUC-ROC = {best_auc:.4f}  (epoch {best_epoch})")

pos_weight = 1.582  (ctrl=7100, dys=4488)


ValueError: Unable to create tensor, you should probably activate padding with 'padding=True' to have batched tensors with the same length.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

epochs_ran = list(range(1, len(train_losses)+1))

ax = axes[0]
ax.plot(epochs_ran, train_losses, label="Train loss", color="steelblue", marker="o")
ax.plot(epochs_ran, val_losses,   label="Val loss",   color="tomato",    marker="o")
ax.axvline(best_epoch, color="green", linestyle="--", alpha=0.6, label=f"Best epoch {best_epoch}")
ax.fill_between(epochs_ran,
                [t - v for t, v in zip(train_losses, val_losses)],
                alpha=0.08, color="purple", label="Train-Val gap")
ax.set_xlabel("Epoch"); ax.set_ylabel("BCE Loss")
ax.set_title("Training vs Validation Loss\n(gap→0 means less overfitting)")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(epochs_ran, val_aucs, label="Val AUC-ROC", color="darkorange", marker="o")
ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5, label="Chance")
ax.axvline(best_epoch, color="green", linestyle="--", alpha=0.6, label=f"Best epoch {best_epoch}")
ax.set_xlabel("Epoch"); ax.set_ylabel("AUC-ROC")
ax.set_title("Validation AUC-ROC per Epoch")
ax.set_ylim(0.4, 1.05); ax.legend(); ax.grid(alpha=0.3)

plt.suptitle("Training Diagnostics — Anti-Overfitting Edition", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: training_curves.png")


In [ ]:
# ── Reload best checkpoint ────────────────────────────────────────────────────
hubert_lora.load_state_dict(torch.load("best_lora_adapter.pt", map_location=device))
model_best = DysarthriaClassifier(hubert_lora).to(device)
model_best.head.load_state_dict(torch.load("best_head.pt", map_location=device))
model_best.eval()

# ── Threshold from val PR curve (selected once, post-training) ────────────────
val_probs, val_true = [], []
with torch.no_grad():
    for iv, am, lbl in val_loader:
        if iv is None:
            continue
        with autocast():
            logits = model_best(iv.to(device), am.to(device))
        val_probs.extend(torch.sigmoid(logits).cpu().float().numpy().flatten())
        val_true.extend(lbl.numpy().flatten())

val_probs = np.array(val_probs)
val_true  = np.array(val_true)

prec, rec, thresh = precision_recall_curve(val_true, val_probs)
f1s      = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-8)
best_idx = int(np.argmax(f1s))
best_thr = float(thresh[best_idx])

print(f"Optimal threshold : {best_thr:.4f}  (val F1 = {f1s[best_idx]:.4f})")
print("This threshold will be applied to the held-out test set.")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.plot(thresh, f1s, color="steelblue", lw=2)
ax.axvline(best_thr, color="red", linestyle="--", label=f"Optimal = {best_thr:.3f}")
ax.set_xlabel("Threshold"); ax.set_ylabel("F1 Score")
ax.set_title("Threshold Sweep — Validation Set"); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(rec, prec, color="darkorange", lw=2)
val_ap = average_precision_score(val_true, val_probs)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title(f"Precision-Recall Curve (Val AP = {val_ap:.3f})")
ax.set_xlim(0, 1); ax.set_ylim(0, 1.05); ax.grid(alpha=0.3)

plt.suptitle("Validation Threshold & PR Curve", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("threshold_pr_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: threshold_pr_curve.png")


In [ ]:
model_best.eval()
test_probs, test_true = [], []

with torch.no_grad():
    for iv, am, lbl in test_loader:
        if iv is None: continue
        with autocast():
            logits = model_best(iv.to(device), am.to(device))
        test_probs.extend(torch.sigmoid(logits).cpu().float().numpy().flatten())
        test_true.extend(lbl.numpy().flatten())

test_probs = np.array(test_probs)
test_true  = np.array(test_true)
test_preds = (test_probs > best_thr).astype(int)

# ── All metrics ───────────────────────────────────────────────────────────────
acc     = accuracy_score(test_true, test_preds)
f1      = f1_score(test_true, test_preds, zero_division=0)
f1_mac  = f1_score(test_true, test_preds, average="macro",  zero_division=0)
f1_wt   = f1_score(test_true, test_preds, average="weighted", zero_division=0)
auc_roc = roc_auc_score(test_true, test_probs)
pr_auc  = average_precision_score(test_true, test_probs)
mcc     = matthews_corrcoef(test_true, test_preds)
cm      = confusion_matrix(test_true, test_preds)

tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn + 1e-8)   # recall for dysarthric class
specificity = tn / (tn + fp + 1e-8)   # recall for control class
ppv         = tp / (tp + fp + 1e-8)   # precision for dysarthric
npv         = tn / (tn + fn + 1e-8)   # precision for control

print("=" * 60)
print("TEST SET RESULTS  (Speaker-Independent, Held-Out)")
print("=" * 60)
print(f"  Threshold              : {best_thr:.4f}")
print(f"  Accuracy               : {acc:.4f}")
print(f"  F1 (dysarthric, pos)   : {f1:.4f}")
print(f"  F1 Macro               : {f1_mac:.4f}")
print(f"  F1 Weighted            : {f1_wt:.4f}")
print(f"  AUC-ROC                : {auc_roc:.4f}")
print(f"  PR-AUC (avg precision) : {pr_auc:.4f}")
print(f"  MCC                    : {mcc:.4f}")
print(f"  Sensitivity (Recall)   : {sensitivity:.4f}")
print(f"  Specificity            : {specificity:.4f}")
print(f"  PPV (Precision)        : {ppv:.4f}")
print(f"  NPV                    : {npv:.4f}")
print(f"  TP={tp}  FP={fp}  TN={tn}  FN={fn}")
print()
print(classification_report(test_true, test_preds,
      target_names=["Control", "Dysarthric"], zero_division=0))


In [ ]:
fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1. Confusion matrix
ax1 = fig.add_subplot(gs[0, 0])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax1,
            xticklabels=["Control", "Dysarthric"],
            yticklabels=["Control", "Dysarthric"])
ax1.set_ylabel("True"); ax1.set_xlabel("Predicted")
ax1.set_title("Confusion Matrix\n(Test Set, Speaker-Independent)")

# 2. ROC Curve
ax2 = fig.add_subplot(gs[0, 1])
fpr_c, tpr_c, _ = roc_curve(test_true, test_probs)
ax2.plot(fpr_c, tpr_c, color="steelblue", lw=2, label=f"AUC = {auc_roc:.3f}")
ax2.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.4, label="Chance")
ax2.set_xlabel("False Positive Rate"); ax2.set_ylabel("True Positive Rate")
ax2.set_title("ROC Curve — Test Set")
ax2.legend(); ax2.grid(alpha=0.3)

# 3. PR Curve (test)
ax3 = fig.add_subplot(gs[0, 2])
prec_t, rec_t, _ = precision_recall_curve(test_true, test_probs)
baseline          = np.sum(test_true) / len(test_true)
ax3.plot(rec_t, prec_t, color="darkorange", lw=2, label=f"AP = {pr_auc:.3f}")
ax3.axhline(baseline, color="gray", linestyle="--", alpha=0.5, label="Baseline")
ax3.set_xlabel("Recall"); ax3.set_ylabel("Precision")
ax3.set_title("Precision-Recall Curve — Test Set")
ax3.legend(); ax3.grid(alpha=0.3)

# 4. Probability distribution
ax4 = fig.add_subplot(gs[1, 0])
ax4.hist(test_probs[test_true == 0], bins=25, alpha=0.6, color="steelblue",
         label="Control (true)", density=True)
ax4.hist(test_probs[test_true == 1], bins=25, alpha=0.6, color="tomato",
         label="Dysarthric (true)", density=True)
ax4.axvline(best_thr, color="green", linestyle="--", lw=2, label=f"Threshold={best_thr:.3f}")
ax4.set_xlabel("Predicted Probability"); ax4.set_ylabel("Density")
ax4.set_title("Score Distribution by True Class")
ax4.legend(); ax4.grid(alpha=0.3)

# 5. Training loss curves
ax5 = fig.add_subplot(gs[1, 1])
epochs_ran = list(range(1, len(train_losses)+1))
ax5.plot(epochs_ran, train_losses, label="Train loss", color="steelblue", marker="o", ms=4)
ax5.plot(epochs_ran, val_losses,   label="Val loss",   color="tomato",    marker="o", ms=4)
ax5.axvline(best_epoch, color="green", linestyle="--", alpha=0.6, label=f"Best={best_epoch}")
ax5.set_xlabel("Epoch"); ax5.set_ylabel("BCE Loss")
ax5.set_title("Loss Curves"); ax5.legend(); ax5.grid(alpha=0.3)

# 6. Metrics bar chart
ax6 = fig.add_subplot(gs[1, 2])
metric_names  = ["Accuracy", "Sensitivity", "Specificity", "F1 (pos)",
                 "AUC-ROC", "PR-AUC", "MCC (norm)"]
metric_values = [acc, sensitivity, specificity, f1,
                 auc_roc, pr_auc, (mcc + 1) / 2]   # normalise MCC to [0,1]
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52",
          "#8172B2", "#937860", "#DA8BC3"]
bars = ax6.barh(metric_names, metric_values, color=colors, alpha=0.8)
ax6.set_xlim(0, 1.1)
ax6.axvline(0.5, color="gray", linestyle="--", alpha=0.4)
for bar, val in zip(bars, metric_values):
    ax6.text(val + 0.01, bar.get_y() + bar.get_height()/2,
             f"{val:.3f}", va="center", fontsize=9)
ax6.set_title("Test Metrics Summary\n(MCC normalised to [0,1])"); ax6.grid(alpha=0.2, axis="x")

plt.suptitle("Dysarthria Detection — Full Evaluation Dashboard (HuBERT+LoRA)",
             fontsize=14, fontweight="bold")
plt.savefig("evaluation_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: evaluation_dashboard.png")


In [ ]:
SEVERITY = {"M01": "Severe-dys", "M05": "Mild-dys", "FC01": "Control", "MC04": "Control"}

res_spk = {}
model_best.eval()

with torch.no_grad():
    for path, true_label in zip(X_test, y_test):
        spk = get_speaker(path)
        sev = SEVERITY.get(spk, "Unknown")
        audio = preprocess_audio(path, augment=False)
        if audio is None:
            continue
        audio = np.array(audio, dtype=np.float32)
        enc = feature_extractor(audio, sampling_rate=SR, return_tensors="pt",
                                padding=True, return_attention_mask=True,
                                max_length=MAX_SAMPLES, truncation=True)
        iv = enc["input_values"].to(device)
        am = enc["attention_mask"].to(device)
        with autocast():
            logit = model_best(iv, am).item()
        prob = torch.sigmoid(torch.tensor(logit)).item()
        pred = int(prob > best_thr)
        key  = f"{spk} ({sev})"
        res_spk.setdefault(key, {"correct": 0, "total": 0, "probs": [], "true": []})
        res_spk[key]["total"]   += 1
        res_spk[key]["correct"] += int(pred == int(true_label))
        res_spk[key]["probs"].append(prob)
        res_spk[key]["true"].append(int(true_label))

print(f"{'Speaker (Severity)':<22}  {'Acc':>6}  {'Avg Prob':>9}  {'N':>5}")
print("-" * 50)
speaker_records = []
for spk_key, r in sorted(res_spk.items()):
    spk_acc  = r["correct"] / r["total"]
    avg_prob = np.mean(r["probs"])
    print(f"{spk_key:<22}  {spk_acc:>6.3f}  {avg_prob:>9.3f}  {r['total']:>5}")
    speaker_records.append({"speaker": spk_key, "accuracy": spk_acc,
                             "avg_prob": avg_prob, "n_samples": r["total"]})

# Per-speaker bar chart
fig, ax = plt.subplots(figsize=(10, 4))
keys  = [r["speaker"] for r in speaker_records]
accs  = [r["accuracy"] for r in speaker_records]
ax.bar(keys, accs, color=["tomato" if "dys" in k.lower() else "steelblue" for k in keys],
       alpha=0.8, edgecolor="black")
ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Accuracy"); ax.set_title("Per-Speaker Accuracy (Test Set)")
for i, (k, v) in enumerate(zip(keys, accs)):
    ax.text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=10)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("per_speaker_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: per_speaker_accuracy.png")


In [ ]:
import csv, json

# ── 1. Metrics JSON ───────────────────────────────────────────────────────────
metrics_dict = {
    "threshold"    : round(best_thr, 6),
    "accuracy"     : round(acc, 6),
    "f1_positive"  : round(f1, 6),
    "f1_macro"     : round(f1_mac, 6),
    "f1_weighted"  : round(f1_wt, 6),
    "auc_roc"      : round(auc_roc, 6),
    "pr_auc"       : round(pr_auc, 6),
    "mcc"          : round(mcc, 6),
    "sensitivity"  : round(sensitivity, 6),
    "specificity"  : round(specificity, 6),
    "ppv"          : round(ppv, 6),
    "npv"          : round(npv, 6),
    "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
    "best_val_auc" : round(best_auc, 6),
    "best_epoch"   : best_epoch,
}

with open("test_metrics.json", "w") as f:
    json.dump(metrics_dict, f, indent=2)

# ── 2. Training history CSV ───────────────────────────────────────────────────
with open("training_history.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=history[0].keys())
    writer.writeheader()
    writer.writerows(history)

# ── 3. Per-speaker CSV ────────────────────────────────────────────────────────
with open("per_speaker_results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=speaker_records[0].keys())
    writer.writeheader()
    writer.writerows(speaker_records)

# ── 4. Per-sample predictions CSV ────────────────────────────────────────────
per_sample = [{"path": p, "speaker": get_speaker(p),
               "true_label": int(y),
               "pred_label": int(p_ > best_thr),
               "prob": round(p_, 6)}
              for p, y, p_ in zip(X_test, y_test, test_probs)]

with open("per_sample_predictions.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=per_sample[0].keys())
    writer.writeheader()
    writer.writerows(per_sample)

# ── 5. Model meta JSON ────────────────────────────────────────────────────────
meta = {
    "backbone"       : "facebook/hubert-base-ls960",
    "lora_rank"      : 8,
    "lora_alpha"     : 16,
    "lora_dropout"   : 0.2,
    "lora_targets"   : ["q_proj", "v_proj"],
    "head_dropout"   : 0.6,
    "weight_decay"   : 0.05,
    "lr"             : 3e-5,
    "label_smoothing": 0.1,
    "best_threshold" : round(best_thr, 6),
    "test_speakers"  : list(TEST_SPEAKERS),
    "val_speakers"   : list(VAL_SPEAKERS),
    "augmentations"  : ["time_stretch", "pitch_shift", "additive_noise", "time_mask"],
}

torch.save(hubert_lora.state_dict(), "dysarthria_lora_final.pt")
torch.save(model_best.head.state_dict(), "dysarthria_head_final.pt")
with open("model_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("=" * 50)
print("All outputs saved:")
print("  test_metrics.json")
print("  training_history.csv")
print("  per_speaker_results.csv")
print("  per_sample_predictions.csv")
print("  model_meta.json")
print("  dysarthria_lora_final.pt")
print("  dysarthria_head_final.pt")
print("  training_curves.png")
print("  threshold_pr_curve.png")
print("  evaluation_dashboard.png")
print("  per_speaker_accuracy.png")
print()
print(json.dumps(metrics_dict, indent=2))


In [ ]:
if device.type == "cuda":
    alloc  = torch.cuda.memory_allocated() / 1e9
    reserv = torch.cuda.memory_reserved()  / 1e9
    total  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM allocated : {alloc:.2f} GB")
    print(f"VRAM reserved  : {reserv:.2f} GB")
    print(f"VRAM total     : {total:.2f} GB")
    print(f"VRAM free      : {total - reserv:.2f} GB")
